# Data Inventory and Quality Audit

## Goal
Establish file-level provenance, schema, temporal coverage, missingness, duplication, and obvious naming inconsistencies before substantive analysis. This notebook intentionally makes no causal claims.

## Context & Methods

### Key assumptions
- Files in `database/` are treated as source exports and are not modified.
- A paired `.txt` file, when present, is treated as the SQL/query provenance record.
- Filename claims such as `since-2008` are checked against observed dates rather than trusted.

In [1]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

def locate_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "database").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "stack_exchange_analysis"
        if (nested / "database").exists() and (nested / "notebooks").exists():
            return nested
    raise FileNotFoundError("Could not locate stack_exchange_analysis project root.")

PROJECT_ROOT = locate_project_root()
DATA_DIR = PROJECT_ROOT / "database"
ANALYSIS_DIR = PROJECT_ROOT / "analysis"
ANALYSIS_DIR.mkdir(exist_ok=True)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from analysis_utils import read_csv_flexible, parse_best_date_column, drop_incomplete_last_period, save_figure

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\Osvaldo\OneDrive\github_oz\stack_analysis\stack_exchange_analysis


### 1. Inventory source files

In [2]:
files = sorted([p for p in DATA_DIR.iterdir() if p.is_file()])
inventory = pd.DataFrame({
    "file": [p.name for p in files],
    "suffix": [p.suffix.lower() for p in files],
    "size_mb": [p.stat().st_size / 1_000_000 for p in files],
})
inventory

,file,suffix,size_mb
0,all-db-names-from-db-system.csv,.csv,0.009746
1,all-db-names-from-db-system.txt,.txt,0.000180
2,cumulative-unanswered-questions-per-month-sinc...,.csv,0.018097
3,cumulative-unanswered-questions-per-month.txt,.txt,0.005446
4,History-Sum-By-Month-Per-Year-of-Votes-by-Tag.csv,.csv,7.246433
5,History-Sum-By-Month-Per-Year-of-Votes-by-Tag.txt,.txt,0.001214
6,new-answers-per-day-since-2008.csv,.csv,0.316708
7,new-answers-per-day-since-2008.txt,.txt,0.000433
8,new-answers-per-month-since-2008.csv,.csv,0.012028
9,new-answers-per-month-since-2008.txt,.txt,0.000553


### 2. Profile CSV files

In [3]:
records = []
for path in sorted(DATA_DIR.glob("*.csv")):
    try:
        df = read_csv_flexible(path)
        date_col, parsed = parse_best_date_column(df)
        record = {
            "file": path.name,
            "rows": len(df),
            "columns": df.shape[1],
            "duplicate_rows": int(df.duplicated().sum()),
            "missing_cells_pct": float(df.isna().mean().mean() * 100),
            "date_column": date_col,
            "date_min": parsed.min() if parsed is not None else pd.NaT,
            "date_max": parsed.max() if parsed is not None else pd.NaT,
            "paired_query": (path.with_suffix('.txt')).exists(),
        }
        records.append(record)
    except Exception as exc:
        records.append({"file": path.name, "error": repr(exc)})

profile = pd.DataFrame(records)
profile

,file,rows,columns,duplicate_rows,missing_cells_pct,date_column,date_min,date_max,paired_query
0,all-db-names-from-db-system.csv,359,1,0,0.000000,NaN,NaT,NaT,True
1,cumulative-unanswered-questions-per-month-sinc...,310,8,0,0.000000,MonthStart,2000-08-01,2026-05-01,False
2,History-Sum-By-Month-Per-Year-of-Votes-by-Tag.csv,92526,8,0,87.500000,NaN,NaT,NaT,True
3,new-answers-per-day-since-2008.csv,8179,3,0,0.000000,Date,2013-12-11,2026-05-14,True
4,new-answers-per-month-since-2008.csv,300,3,0,0.000000,Date,2013-12-01,2026-05-01,True
5,new-question-activity-per-day-since-2008.csv,3891,2,0,0.000000,Date,2013-12-11,2026-05-16,True
6,new-question-activity-per-month-since-2008.csv,147,2,0,0.000000,Date,2013-12-01,2026-05-01,True
7,new-questions-per-day-by-sites-since-2008.csv,6499,22,0,0.000000,Day,2008-07-31,2026-05-16,True
8,tags-db.csv,2704,7,0,38.767963,NaN,NaT,NaT,True
9,user-metrics-by-site-by-month-all-time.csv,3280,44,0,0.491269,CreationMonth,2008-07-01,2026-05-01,True


### 3. Flag temporal naming inconsistencies

In [4]:
checks = profile.copy()
checks["claims_since_2008"] = checks["file"].str.contains("since-2008", case=False, na=False)
checks["observed_start_year"] = pd.to_datetime(checks["date_min"], errors="coerce").dt.year
checks["start_year_mismatch"] = checks["claims_since_2008"] & checks["observed_start_year"].notna() & (checks["observed_start_year"] > 2008)
checks.loc[checks["claims_since_2008"], ["file", "observed_start_year", "start_year_mismatch"]]

,file,observed_start_year,start_year_mismatch
1,cumulative-unanswered-questions-per-month-sinc...,2000.0,False
3,new-answers-per-day-since-2008.csv,2013.0,True
4,new-answers-per-month-since-2008.csv,2013.0,True
5,new-question-activity-per-day-since-2008.csv,2013.0,True
6,new-question-activity-per-month-since-2008.csv,2013.0,True
7,new-questions-per-day-by-sites-since-2008.csv,2008.0,False


### 4. Inspect query provenance

In [5]:
query_records = []
for path in sorted(DATA_DIR.glob("*.txt")):
    text = path.read_text(encoding="utf-8", errors="replace")
    lower = text.lower()
    query_records.append({
        "query_file": path.name,
        "chars": len(text),
        "mentions_stackoverflow": "stackoverflow" in lower,
        "mentions_pt_endpoint": "data.stackexchange.com/pt/" in lower,
        "mentions_postswithdeleted": "postswithdeleted" in lower,
        "mentions_posts": " posts" in lower or "..posts" in lower,
    })
query_audit = pd.DataFrame(query_records)
query_audit

,query_file,chars,mentions_stackoverflow,mentions_pt_endpoint,mentions_postswithdeleted,mentions_posts
0,all-db-names-from-db-system.txt,172,True,False,False,False
1,cumulative-unanswered-questions-per-month.txt,5281,False,True,False,True
2,History-Sum-By-Month-Per-Year-of-Votes-by-Tag.txt,1169,False,False,False,True
3,new-answers-per-day-since-2008.txt,415,False,False,True,True
4,new-answers-per-month-since-2008.txt,541,True,False,True,True
5,new-question-activity-per-day-since-2008.txt,220,False,False,False,True
6,new-question-activity-per-month-since-2008.txt,384,True,False,False,True
7,new-questions-per-day-by-sites-since-2008.txt,5383,True,False,False,True
8,tags-db.txt,18,False,False,False,False
9,user-metrics-by-site-by-month-all-time.txt,4005,True,False,False,False


### 5. Export audit tables

In [6]:
inventory.to_csv(ANALYSIS_DIR / "data_file_inventory.csv", index=False)
profile.to_csv(ANALYSIS_DIR / "data_quality_profile.csv", index=False)
query_audit.to_csv(ANALYSIS_DIR / "query_provenance_audit.csv", index=False)
print("Audit tables written to analysis/.")

Audit tables written to analysis/.


## Takeaways
Use the generated audit tables to resolve source/site ambiguities before combining datasets. In particular, do not infer site identity from filenames alone.